# YOLO Bukukia Training Pipeline — v3.1 (GCP VM, DPI-Aware)

This notebook is for execution on a **GCP VM instance** (not Google Colab).
Run cells sequentially in JupyterLab or VS Code on the VM.

## Differences from Colab version
- No Google Drive mount
- Project files are on the VM's local disk
- GPU accessed directly via CUDA
- System packages installed via `apt` if needed

## Pipeline Overview

| Step | Section | Script |
|---|---|---|
| 0 | System Check + Setup | — |
| 1 | Configuration | — |
| 2 | DPI Preprocessing (600→96) | inline |
| 3 | Training (Focal Loss) | `train_2.py` |
| 4 | Model Evaluation | `test_model.py` |
| 5 | Fine-Tuning *(optional)* | `fine_tune.py` |
| 6 | Manual / Advanced Tools | various |

---
## 0. System Check & Setup

Run this once to verify GPU availability and install required packages.

In [ ]:
# ── GPU Check ──
!nvidia-smi

In [ ]:
# ── Install system dependencies (run once) ──
!apt-get install -y libgl1-mesa-glx libglib2.0-0 -q

# ── Install Python dependencies ──
!pip install ultralytics>=8.3.0 opencv-python-headless matplotlib pandas Pillow -q

print("Done.")

In [ ]:
import os, sys
from pathlib import Path

# ── Set project directory ──
# Change this to where the project is cloned/uploaded on your VM
PROJECT_DIR = Path('')

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_DIR}\n"
                            f"Please update PROJECT_DIR to your actual path.")

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
print(f"Working dir : {os.getcwd()}")
print(f"Python      : {sys.executable}")

# Verify key scripts exist
scripts = ['train_2.py', 'test_model.py', 'config.py', 'iou_callback.py',
           'evaluate_iou_full.py', 'fine_tune.py']
for s in scripts:
    path = PROJECT_DIR / 'scripts' / s
    status = '✅' if path.exists() else '❌ MISSING'
    print(f"  scripts/{s}: {status}")

In [ ]:
# ── Verify config and print environment ──
from scripts.config import print_config
print_config()

---
## 1. Configuration

Set the model, training parameters, and DPI settings here.

| Parameter | Description | Default |
|---|---|---|
| `YOLO_MODEL` | Model checkpoint name or path | `yolo26n.pt` |
| `YOLO_EPOCHS` | Number of training epochs | `50` |
| `YOLO_CONF` | Confidence threshold | `0.25` |
| `ORIGINAL_DPI` | Resolution of production pages | `600` |
| `TRAINING_DPI` | Resolution for YOLO training | `96` |

In [ ]:
import os

# ── Model Settings ──
os.environ['YOLO_MODEL']  = 'yolo26n.pt'
os.environ['YOLO_EPOCHS'] = '50'
os.environ['YOLO_CONF']   = '0.25'

# ── DPI Settings ──
os.environ['ORIGINAL_DPI'] = '600'
os.environ['TRAINING_DPI'] = '96'

# ── Verify ──
original_dpi = int(os.environ['ORIGINAL_DPI'])
training_dpi = int(os.environ['TRAINING_DPI'])
scale_factor = original_dpi / training_dpi

print(f"Model        : {os.environ['YOLO_MODEL']}")
print(f"Epochs       : {os.environ['YOLO_EPOCHS']}")
print(f"Original DPI : {original_dpi}")
print(f"Training DPI : {training_dpi}")
print(f"Scale Factor : {scale_factor}x")

---
## 2. DPI Preprocessing — Downsample 600 → 96 DPI

Convert 600 DPI training images to 96 DPI for YOLO training.

| | Path |
|---|---|
| **Input** | `input_files/raw_images/*.png` — Original 600 DPI labeled scans |
| **Output** | `input_files/raw_images_96dpi/*.png` — Downsampled 96 DPI copies |

> YOLO labels use **normalized coordinates (0–1)**, so they remain valid after resizing — no label changes needed.

In [ ]:
from pathlib import Path
from PIL import Image
import os

original_dpi = int(os.environ.get('ORIGINAL_DPI', 600))
training_dpi = int(os.environ.get('TRAINING_DPI', 96))
scale_factor = original_dpi / training_dpi  # 6.25

RAW_IMAGES      = Path('input_files/raw_images')
DOWNSAMPLED_DIR = Path('input_files/raw_images_96dpi')
DOWNSAMPLED_DIR.mkdir(parents=True, exist_ok=True)

extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp'}
images = [f for f in RAW_IMAGES.iterdir() if f.suffix.lower() in extensions]

print(f"Found {len(images)} images in {RAW_IMAGES}")
print(f"Downsampling {original_dpi} DPI → {training_dpi} DPI (÷{scale_factor})")
print(f"Output : {DOWNSAMPLED_DIR}")
print("-" * 50)

skipped = 0
processed = 0

for img_path in sorted(images):
    out_path = DOWNSAMPLED_DIR / img_path.name

    if out_path.exists() and out_path.stat().st_mtime >= img_path.stat().st_mtime:
        skipped += 1
        continue

    img = Image.open(img_path)
    new_w = int(img.width / scale_factor)
    new_h = int(img.height / scale_factor)
    img_resized = img.resize((new_w, new_h), Image.LANCZOS)
    img_resized.save(out_path)
    processed += 1

    if processed <= 3:
        print(f"  {img_path.name}: {img.width}×{img.height} → {new_w}×{new_h}")

print(f"\nDone! Processed: {processed} | Skipped (cached): {skipped}")

---
## 3. Training — `train_2.py` (Focal Loss)

Trains with **Focal Loss** and **IoU callback** on 96 DPI images.

| | Path / Description |
|---|---|
| **Input** | `input_files/raw_images_96dpi/` — 96 DPI images (from Step 2) |
| **Input** | `input_files/export/labels/*.txt` — YOLO labels |
| **Input** | `input_files/export/classes.txt` — Class names |
| **Output** | `results/runs/train/weights/best.pt` — Best trained model |
| **Output** | `results/runs/train/training_analysis.png` — Dashboard |
| **Output** | `results/runs/train/iou_log.csv` — Per-epoch IoU |

### Arguments

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | config | Model name or path |
| `--epochs` | int | config | Number of training epochs |
| `--imgsz` | int | `640` | Image size |
| `--batch` | int | `-1` | Batch size (`-1` = auto) |
| `--limit` | int | `0` | Limit total images (0 = all) |
| `--fraction` | float | `0.0` | Use % of data |
| `--images-dir` | str | `raw_images` | Custom image folder |
| `--split-dir` | str | — | Pre-split dataset (skips auto-split) |
| `--fl-gamma` | float | `1.5` | Focal Loss gamma (0.0 = disable) |
| `--fl-alpha` | float | `0.25` | Focal Loss alpha |

In [ ]:
# ── Training with Focal Loss on 96 DPI images ──

!python scripts/train_2.py \
    --model $YOLO_MODEL \
    --epochs $YOLO_EPOCHS \
    --imgsz 640 \
    --batch -1 \
    --images-dir input_files/raw_images_96dpi \
    --fl-gamma 1.5 \
    --fl-alpha 0.25

### Alternative: Train with Pre-Split Dataset

In [ ]:
# Uncomment and adjust:

# !python scripts/train_2.py \
#     --model $YOLO_MODEL \
#     --epochs $YOLO_EPOCHS \
#     --imgsz 640 \
#     --batch -1 \
#     --fl-gamma 1.5 \
#     --fl-alpha 0.25 \
#     --split-dir input_files/dataset_fixpage96

---
## 4. Model Evaluation — `test_model.py`

Evaluate on a separate test set.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--conf` | float | `0.25` | Confidence threshold |
| `--model` | str | auto-detected | Model file (.pt) |
| `--data` | str | `test-dataset` | Test images directory |

In [ ]:
!python scripts/test_model.py \
    --conf  $YOLO_CONF \
    --model results/runs/train/weights/best.pt \
    --data  input_files/test-dataset

---
## 5. Fine-Tuning *(Optional)* — `fine_tune.py`

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | auto-detected | Model .pt |
| `--iterations` | int | `30` | Tuning iterations |
| `--epochs` | int | `30` | Epochs per trial |
| `--imgsz` | int | `640` | Image size |

In [ ]:
# Uncomment to run:

# !python scripts/fine_tune.py \
#     --model results/runs/train/weights/best.pt \
#     --iterations 30 \
#     --epochs 30 \
#     --imgsz 640

---
---
# 6. Manual / Advanced Tools

Standalone utility scripts. Each section is independent and ready to run.

---
### 6.1 Evaluate IoU — Validation Set (`evaluate_iou_full.py`)

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | **required** | Trained model (.pt) |
| `--data` | str | **required** | Validation dataset directory |
| `--out` | str | `.` | Output directory |
| `--conf` | float | `0.25` | Confidence threshold |
| `--iou-thres` | float | `0.5` | IoU threshold |

In [ ]:
!python scripts/evaluate_iou_full.py \
    --model results/runs/train/weights/best.pt \
    --data  results/dataset \
    --out   results/iou_results

---
### 6.2 Evaluate IoU — Training Set (`evaluate_iou_full_train.py`)

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | **required** | Trained model (.pt) |
| `--data` | str | **required** | Dataset directory |
| `--out` | str | `.` | Output directory |

In [ ]:
!python scripts/evaluate_iou_full_train.py \
    --model results/runs/train/weights/best.pt \
    --data  results/dataset \
    --out   results/iou_results_train

---
### 6.3 Predict to Labels (`predict_to_labels.py`)

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | **required** | Trained model (.pt) |
| `--source` | str | **required** | Images directory |
| `--out` | str | `predicted_labels` | Output directory |
| `--conf` | float | `0.25` | Confidence threshold |
| `--imgsz` | int | `640` | Image size |

In [ ]:
!python scripts/predict_to_labels.py \
    --model  results/runs/train/weights/best.pt \
    --source input_files/raw_images_96dpi \
    --out    results/predicted_labels \
    --conf   0.25

---
### 6.4 Extract Label Details (`extract_label_details.py`)

| Argument | Type | Default | Description |
|---|---|---|---|
| `--labels-dir` | str | **required** | YOLO label .txt files |
| `--classes-txt` | str | **required** | Path to `classes.txt` |
| `--out` | str | `label_details.csv` | Output CSV |

In [ ]:
!python scripts/extract_label_details.py \
    --labels-dir  input_files/export/labels \
    --classes-txt input_files/export/classes.txt \
    --out         results/label_details.csv

---
### 6.5 Manual Hyperparameter Tuning (`fine_tune_tuningmanual.py`)

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | auto-detected | Model .pt |
| `--epochs` | int | `30` | Epochs per trial |
| `--imgsz` | int | `640` | Image size |
| `--batch` | int | `16` | Batch size |
| `--rounds` | int | `2` | Descent rounds |

In [ ]:
# !python scripts/fine_tune_tuningmanual.py \
#     --model results/runs/train/weights/best.pt \
#     --epochs 30 \
#     --imgsz 640 \
#     --batch 16 \
#     --rounds 2

---
### 6.6 Copy Images from Labels (`copy_images_from_labels_colab.py`)

| Argument | Type | Default | Description |
|---|---|---|---|
| `--labels-dir` | str | **required** | Label .txt files |
| `--images-dir` | str | **required** | Source images |
| `--out` | str | **required** | Destination |

In [ ]:
!python scripts/copy_images_from_labels_colab.py \
    --labels-dir input_files/export/labels \
    --images-dir input_files/raw_images_96dpi \
    --out        input_files/matched_images

---
### 6.7 Filter Images by CSV (`filter_images_colab.py`)

| Argument | Type | Default | Description |
|---|---|---|---|
| `--csv` | str | **required** | CSV with `filename` column |
| `--images-dir` | str | **required** | Source images |
| `--out` | str | **required** | Destination |

In [ ]:
!python scripts/filter_images_colab.py \
    --csv        results/filter_list.csv \
    --images-dir input_files/raw_images_96dpi \
    --out        input_files/filtered_images

---
### 6.8 Filter Labels by CSV (`filter_labels_colab.py`)

| Argument | Type | Default | Description |
|---|---|---|---|
| `--csv` | str | **required** | CSV with `filename` column |
| `--labels-dir` | str | **required** | Source labels |
| `--out` | str | **required** | Destination |

In [ ]:
!python scripts/filter_labels_colab.py \
    --csv        results/filter_list.csv \
    --labels-dir input_files/export/labels \
    --out        input_files/filtered_labels

---
### 6.9 Filter Labels by Segment (`filter_labels_by_segment_colab.py`)

| Argument | Type | Default | Description |
|---|---|---|---|
| `--labels-dir` | str | **required** | Source labels |
| `--out` | str | **required** | Destination |
| `--classes` | int[] | **required** | Class IDs to keep |

In [ ]:
!python scripts/filter_labels_by_segment_colab.py \
    --labels-dir input_files/export/labels \
    --out        input_files/filtered_by_segment \
    --classes 0 1 2

---
### 6.10 Move/Copy Random Images (`move_random_images_colab.py`)

| Argument | Type | Default | Description |
|---|---|---|---|
| `--images-dir` | str | **required** | Source images |
| `--labels-dir` | str | — | Source labels |
| `--out` | str | **required** | Destination |
| `--count` | int | **required** | Number to select |
| `--move` | flag | `False` | Move instead of copy |
| `--seed` | int | `42` | Random seed |

In [ ]:
!python scripts/move_random_images_colab.py \
    --images-dir input_files/raw_images_96dpi \
    --labels-dir input_files/export/labels \
    --out        input_files/test-dataset \
    --count 20 \
    --seed 42